# EDA - Customer Churn

Exploración rápida del dataset histórico antes de armar el pipeline.
Esto es solo para entender los datos; el entrenamiento real corre desde
`src/training/train.py`, no desde acá.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/customer_churn_historical.csv")
df.shape

(7043, 21)

In [2]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [3]:
df.isna().sum().sort_values(ascending=False).head(10)

TotalCharges      26
gender             0
SeniorCitizen      0
Partner            0
customerID         0
Dependents         0
tenure             0
MultipleLines      0
PhoneService       0
OnlineSecurity     0
dtype: int64

`TotalCharges` viene como texto y tiene vacíos en los clientes con
`tenure == 0` (todavía no se les generó ni una factura). Se resuelve
convirtiendo a numérico y dejando que el imputer del pipeline se
encargue del resto.

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges
95,0,24.03,NaN
781,0,83.01,NaN
854,0,68.81,NaN
867,0,75.01,NaN
882,0,74.50,NaN
1013,0,24.19,NaN
1172,0,89.15,NaN
1842,0,89.85,NaN
2112,0,59.50,NaN
2346,0,84.69,NaN


In [5]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.736334
Yes    0.263666
Name: proportion, dtype: float64

Dataset desbalanceado (~26% churn). Con esto en mente, accuracy sola no
alcanza para elegir modelo: hay que mirar recall/precision de la clase
positiva y el trade-off de un falso negativo (cliente que se va y el
modelo dice que no).

In [6]:
df.groupby("Contract")["Churn"].value_counts(normalize=True).unstack()

Churn,No,Yes
Contract,,
Month-to-month,0.612700,0.387300
One year,0.850264,0.149736
Two year,0.910256,0.089744


In [7]:
df.groupby("InternetService")["Churn"].value_counts(normalize=True).unstack()

Churn,No,Yes
InternetService,,
DSL,0.800343,0.199657
Fiber optic,0.597472,0.402528
No,0.923822,0.076178


In [8]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7017.000000
mean,35.168394,68.168503,2312.077586
std,18.901478,24.980659,1573.967858
min,0.000000,18.000000,0.000000
25%,20.000000,55.360000,986.630000
50%,35.000000,73.910000,2013.200000
75%,51.000000,88.000000,3452.850000
max,72.000000,114.410000,7761.340000
